# SpeechT5 + WavLM Fine-Tuning

This notebook fine-tunes **SpeechT5** (`microsoft/speecht5_vc`) to accept **WavLM hidden states**
as encoder input instead of raw audio, enabling a WavLM-based feature-space speech translation pipeline.

## Architecture
```
EN raw audio  ──► [WavLM Encoder (frozen)]  ──► EN hidden states (Seq, 768)
                                                        │
                                 [SpeechT5 Transformer (fine-tuned)] ──► DE mel spectrogram
                                                        │
                                             [HiFi-GAN Vocoder]  ──► DE audio
```

## Required Pre-Step
The preprocessed WavLM dataset must exist at:
```
datasets/processed_wavlm_en_de_v1/
    en/   ← WavLM hidden states (Seq, 768)
    de/   ← WavLM hidden states (Seq, 768)
```
Run `preprocess_wavlm.py` if the dataset is not yet generated.

> **Note on the target representation:**  
> The dataset stores WavLM features for both sides. The `SpeechT5WavLMDataset`
> class in `model.py` handles the fallback to mel-spectrogram extraction if the
> target is raw audio. The fine-tuning loss is computed against SpeechT5's
> mel-spectrogram decoder output — this is what teaches the model to bridge
> the WavLM representation into something the trained decoder can process.


## 1. Setup & Imports

In [2]:
import os
import sys

# Add project root to Python path so that dataset_loader and encoders are importable.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

from model import SpeechT5WavLM
import dataset_loader

print(f"Project root: {project_root}")
print(f"Datasets directory: {dataset_loader.DATASETS_DIR}")

Project root: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model
Datasets directory: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets


## 2. Configuration

Edit the constants below to control training behaviour.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Path to the preprocessed WavLM dataset produced by preprocess_wavlm.py.
# ─────────────────────────────────────────────────────────────────────────────
PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_wavlm_en_de_v2")

# ─────────────────────────────────────────────────────────────────────────────
# Training hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS        = 10       # Total training epochs
BATCH_SIZE    = 1        # Keep small — WavLM sequences are variable length
LEARNING_RATE = 5e-6     # Low LR for stable fine-tuning of pre-trained weights

# ─────────────────────────────────────────────────────────────────────────────
# Output checkpoint directory
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "speecht5_wavlm_en_de_v2"

print(f"Preprocessed data: {PREPROCESSED_PATH}")
print(f"Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"Checkpoint will be saved to: {CHECKPOINT_DIR}")

Preprocessed data: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2
Epochs: 10  |  Batch size: 1  |  LR: 5e-06
Checkpoint will be saved to: speecht5_wavlm_en_de_v2


## 3. Verify Preprocessed Dataset

Confirm the dataset exists and inspect a few sample feature shapes before starting training.

In [4]:
from datasets import load_from_disk
import numpy as np

assert os.path.isdir(PREPROCESSED_PATH), (
    f"Preprocessed dataset not found at '{PREPROCESSED_PATH}'.\n"
    "Run preprocess_wavlm.py first."
)

# Load the unified paired dataset (v2 format)
paired_ds = load_from_disk(PREPROCESSED_PATH)

print(f"\nTotal pairs: {len(paired_ds)}")
print(f"Dataset columns: {paired_ds.column_names}")

print("\nSample feature shapes (first 5 pairs):")
for i in range(min(5, len(paired_ds))):
    src = np.array(paired_ds[i]['input_values'])
    tgt = np.array(paired_ds[i]['labels'])
    
    # Source should be WavLM (Seq, 768)
    if src.ndim == 1 and src.size % 768 == 0:
        src = src.reshape(-1, 768)
    
    # Target should be Mel (T, 80)
    # If this fails, the sample contains raw audio or is corrupted
    if tgt.ndim == 1:
        if tgt.size % 80 == 0:
            tgt = tgt.reshape(-1, 80)
            target_info = f"Mel: {tgt.shape}"
        else:
            target_info = f"INVALID (Size {tgt.size} not divisible by 80. Likely raw audio!)"
    else:
        target_info = f"Shape: {tgt.shape}"
    
    print(f"  [{i}] Source (WavLM): {src.shape if src.ndim > 1 else f'Flat {src.size}'} | Target: {target_info}")



Total pairs: 13438
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels']

Sample feature shapes (first 5 pairs):
  [0] Source (WavLM): (212, 768) | Target: Shape: (259, 80)
  [1] Source (WavLM): (145, 768) | Target: Shape: (176, 80)
  [2] Source (WavLM): (217, 768) | Target: Shape: (273, 80)
  [3] Source (WavLM): (102, 768) | Target: Shape: (151, 80)
  [4] Source (WavLM): (192, 768) | Target: Shape: (349, 80)


## 4. Initialise the Model

`SpeechT5WavLM` loads:
- **SpeechT5** (`microsoft/speecht5_vc`) — the transformer to fine-tune
- **HiFi-GAN** (`microsoft/speecht5_hifigan`) — the vocoder (frozen, used at inference)

WavLM itself is **not** loaded here; it was already used during preprocessing to produce the
hidden states stored in the dataset. It is only needed again at inference time
(handled by `run_inference`).

In [5]:
model = SpeechT5WavLM()
print(f"\nModel loaded on device: {model.device}")

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda

Model loaded on device: cuda


## 5. Extract Target Speaker Embedding

The SpeechT5 decoder needs an **x-vector speaker embedding** to condition its output voice.

`get_speaker_embedding` streams one sample from Google FLEURS for the target language and
computes the x-vector using the SpeechBrain classifier. The embedding is saved alongside
the model checkpoint.

In [6]:
model.get_speaker_embedding(target_lang="de")

Initializing X-Vector classifier for embedding extraction...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


Extracting target speaker embedding...
Embedding extracted successfully.


## 6. Fine-Tune

Training will:
1. Load the preprocessed WavLM dataset.
2. Freeze SpeechT5's CNN feature encoder (only transformer layers are trained).
3. Feed WavLM hidden states directly into SpeechT5's transformer encoder.
4. Compute the spectrogram loss against the decoder's mel output.
5. Save a checkpoint every 5 epochs. On `KeyboardInterrupt` progress is saved
   automatically to `speecht5_wavlm_interrupted`.

In [ ]:
model.fine_tune(
    preprocessed_path=PREPROCESSED_PATH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

Starting WavLM+SpeechT5 fine-tuning (Hybrid Architecture).
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v2...
Loaded 13438 aligned (source, target) pairs.
Dataset columns: ['id', 'language', 'gender', 'transcription', 'input_values', 'labels']
Freezing SpeechT5 CNN feature encoder (not used in hybrid path).


Epoch 1/10: 100%|██████████| 13438/13438 [18:15<00:00, 12.27it/s, loss=16.2905, l1_pre=8.2623, l1_post=8.0281]  


Epoch 1/10  Avg Loss: 15.9156


Epoch 2/10:  94%|█████████▍| 12599/13438 [16:57<01:04, 13.03it/s, loss=15.6996, l1_pre=7.9714, l1_post=7.7280]  

## 7. Save the Final Checkpoint

In [ ]:
model.save(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")

## 8. Quick Inference Test

Load a raw EN clip from the raw dataset and run it through the full pipeline
(WavLM → fine-tuned SpeechT5 transformer → HiFi-GAN) to hear if the model is learning.

In [ ]:
from IPython.display import Audio, display
import numpy as np

model.load(CHECKPOINT_DIR)
# Load a raw EN sample from the seamless_align dataset for inference.
print("Loading a raw EN sample for inference test...")
raw = dataset_loader.load_data(
    lang=[source_lang],
    split="train",
    dataset=["fleurs"],
    num_samples=100,
)
sample = raw[source_lang][0]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f"Input audio: {len(audio_array)/sr:.2f}s @ {sr}Hz")
print("Original EN audio:")
display(Audio(data=audio_array, rate=sr))

# Run the full pipeline: raw audio → WavLM → fine-tuned SpeechT5 → HiFi-GAN → audio
print("\nRunning fine-tuned inference...")
result = model.run_inference(
    audio_array=audio_array,
    sampling_rate=sr,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=2.0,
)

out_audio = result['audio']['array']
out_sr    = result['audio']['sampling_rate']

print(f"Output audio: {len(out_audio)/out_sr:.2f}s @ {out_sr}Hz")
print("\nGenerated DE audio (after fine-tuning):")
display(Audio(data=out_audio, rate=out_sr))

## Bonus: Resume Training from a Checkpoint

If training was interrupted, load the saved checkpoint and continue.

In [ ]:
# ── Uncomment to resume from a saved checkpoint ──────────────────────────────
# RESUME_CHECKPOINT = "speecht5_wavlm_interrupted"   # or "checkpoint_epoch_5", etc.
# model_resume = SpeechT5WavLM()
# model_resume.load(RESUME_CHECKPOINT)
# model_resume.fine_tune(
#     preprocessed_path=PREPROCESSED_PATH,
#     epochs=EPOCHS,
#     learning_rate=LEARNING_RATE,
#     batch_size=BATCH_SIZE,
# )